# 🔴 Solution: Conv2D Backward (NumPy)

In [ ]:
import numpy as np

In [ ]:
# ✅ SOLUTION

def conv2d_backward(dout, x, w, stride=1, padding=0):
    # dout: (N, F, H_out, W_out);  x: (N, C, H, W);  w: (F, C, KH, KW)
    N, C, H, W = x.shape
    F, _, KH, KW = w.shape
    _, _, H_out, W_out = dout.shape
    s, p = stride, padding

    x_pad = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p))) if p > 0 else x

    dx_pad = np.zeros_like(x_pad, dtype=np.float64)
    dw = np.zeros_like(w, dtype=np.float64)

    # The bias is broadcast over batch and space -> sum the gradient back over those dims
    db = dout.sum(axis=(0, 2, 3))

    for i in range(H_out):
        for j in range(W_out):
            d = dout[:, :, i, j]                                      # (N, F)
            patch = x_pad[:, :, i * s:i * s + KH, j * s:j * s + KW]    # (N, C, KH, KW)

            # dw: contract over the batch -> (F, C, KH, KW). Weight sharing means we accumulate.
            dw += np.tensordot(d, patch, axes=([0], [0]))

            # dx: scatter the gradient back through the same window. Overlaps must add up.
            dx_pad[:, :, i * s:i * s + KH, j * s:j * s + KW] += np.tensordot(d, w, axes=([1], [0]))

    # Crop the padded border so dx has the shape of x
    dx = dx_pad[:, :, p:p + H, p:p + W]

    return dx, dw, db

In [ ]:
# Verify
np.random.seed(0)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(4, 3, 3, 3)
dout = np.random.randn(2, 4, 6, 6)

dx, dw, db = conv2d_backward(dout, x, w, stride=1, padding=0)
print("dx:", dx.shape, " dw:", dw.shape, " db:", db.shape)
print("db == dout.sum:", np.allclose(db, dout.sum(axis=(0, 2, 3))))

xs = np.zeros((1, 1, 4, 4))
ws = np.ones((1, 1, 2, 2))
dxs, _, _ = conv2d_backward(np.ones((1, 1, 3, 3)), xs, ws, stride=1, padding=0)
print("corner pixel  :", dxs[0, 0, 0, 0], "(expect 1.0)")
print("interior pixel:", dxs[0, 0, 1, 1], "(expect 4.0)")

In [ ]:
from torch_judge import check
check("numpy_conv2d_backward")